# Stage 2 Notebook 73 - Speed test: NB62 recipe + batch=32 + workers=6 + torch.compile

**Priority 1 of the speed/pretrained/CULane plan.** NB70 took 10h, NB71 took 8h at batch=8, workers=2, no torch.compile. RTX Pro 6000 was 5% utilized. This run tests the same lane recipe (NB62 = anchor + cls_separate_path + VFL + bb_throttle) with:

- `batch_size: 8 -> 32` (4x throughput per step)
- `workers: 2 -> 6` (less dataloader idle)
- `lr0: 2e-4 -> 4e-4` (sqrt(4)x scaling for batch increase)
- `torch.compile(reduce-overhead)` (+20-40% on transformer)
- `persistent_workers=True` + `prefetch_factor=4`

Expected: full 70K / 32 = 2188 iters/epoch (vs 8750), ~3x speedup per epoch. 12 epochs should drop from 14h (NB62) to ~2-3h.

If metrics match NB62 (matched_iou ~0.55, decoded_f1 ~0.06): speed flags are safe defaults for all future runs and we add them to every yaml.
If metrics regress: LR scaling was wrong, revert to lr0=2e-4 or use gradient accumulation.

### Run mode
1. Smoke.
2. 12 epochs full 70K. ~2-3 hr (was ~14 hr at batch=8).

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp71_rmt_gca_anchor_cls_sep_vfl_speed_batch32_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp71_rmt_gca_anchor_cls_sep_vfl_speed_batch32_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp71_rmt_gca_anchor_cls_sep_vfl_speed_batch32_joint_smoke.log
OK exp71_rmt_gca_anchor_cls_sep_vfl_speed_batch32_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.1793 det_loss=3.2866 grad_cos=-0.0387 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5000896453857422, 'gate/lane_mean': 0.5013031363487244, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [ ]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp71_rmt_gca_anchor_cls_sep_vfl_speed_batch32_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 8
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full12_speed'
    EPOCHS = 12
    BATCH_SIZE = 32
    LIMIT_TRAIN = None
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
    # SPEED FLAGS:
    '--workers', '6',
    '--prefetch-factor', '4',
    '--torch-compile',
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('BATCH_SIZE:', BATCH_SIZE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
BATCH_SIZE: 32
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp71_rmt_gca_anchor_cls_sep_vfl_speed_batch32_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp71_rmt_gca_anchor_cls_sep_vfl_speed_batch32_joint_full12_speed --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp71_rmt_gca_anchor_cls_sep_vfl_speed_batch32_joint_full12_speed.tar --epochs 12 --batch-size 32 --limit-val 1000 --force-extract --print-every 50 --workers 6 --prefetch-factor 4 --torch-compile
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp71_rmt_gca_anchor_cls_sep_vfl_speed_batch32_joint_full12_speed.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp71_rmt_gca_anchor_cls_sep_vfl_speed_batch32_joint_full12_speed_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/

0

## What to watch in NB73 (speed test)

Reference NB62 (batch=8, 12 ep full, ~14 hr): matched_iou=0.55, decoded_f1=0.06, gap=0.024, val_map50=0.

Pass criteria:
- **Wall-clock <= 4 hr** (5x speedup vs NB62).
- `val/matched_line_iou >= 0.50` -- preserve NB62 geometry within noise.
- `val/lane/decoded_f1 >= 0.05` -- preserve NB62 cls.
- `[loader]` log line shows workers=6 persistent=True prefetch=4.
- `[speed] torch.compile(...)` log line at start.
- `peak_mem_mb` ~ 30-40 GB (vs NB62's 10 GB).

If pass: these flags become default for ALL future configs.
If matched_iou drops > 0.05: LR scaling was wrong. Switch to gradient accumulation (--batch-size 8 --grad-accum-steps 4) to simulate batch=32 with NB62's LR.